# COLAB ROME Llama2 Sanity

Llama2-7B + ROME small sanity run (3 rounds).


## Runtime
- Python 3
- T4 GPU
- Need HuggingFace access to `meta-llama/Llama-2-7b-hf` and run `huggingface-cli login`.


In [ ]:
# Purpose: Mount Drive and prepare workspace.
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs('/content/work', exist_ok=True)
%cd /content



In [ ]:
# Purpose: Clone EasyEdit and install dependencies.
!rm -rf /content/EasyEdit
!git clone --depth 1 https://github.com/zjunlp/EasyEdit.git

!pip -q install -U pip setuptools wheel
!pip -q install "PyYAML==6.0.2"
!pip -q install -r /content/EasyEdit/requirements.txt



In [ ]:
# Purpose: Copy scripts/data from your Drive project.
import os, shutil, glob

hits = glob.glob('/content/drive/MyDrive/**/scripts/run_edit_fairness_rounds.py', recursive=True)
assert hits, 'Cannot find project scripts in MyDrive'
ROOT = hits[0].split('/scripts/run_edit_fairness_rounds.py')[0]
print('ROOT =', ROOT)

%cd /content/work
!rm -rf /content/work/*

for fn in ['scripts/run_edit_fairness_rounds.py','scripts/run_fairness_eval.py','scripts/calc_drift_metrics.py','scripts/plot_drift.py']:
    shutil.copy(os.path.join(ROOT, fn), '/content/work')
for fn in ['data/edits_bias_stress_60.json','data/crows_pairs_sample.jsonl','data/bbq_sample.jsonl']:
    shutil.copy(os.path.join(ROOT, fn), '/content/work')

print('copied ok')



In [ ]:
# Purpose: Patch GRACE class name mismatch if present.
p = '/content/work/run_edit_fairness_rounds.py'
txt = open(p, 'r', encoding='utf-8').read()
txt = txt.replace('GRACEHyperParams', 'GraceHyperParams')
txt = txt.replace('"GRACE": GRACEHyperParams', '"GRACE": GraceHyperParams')
open(p, 'w', encoding='utf-8').write(txt)
print('patched')



In [ ]:
# Purpose: Login HuggingFace (needed for Llama2 model access).
# Run and paste your HF token in prompt.
!huggingface-cli login



In [ ]:
# Purpose: Write ROME hparams for Llama2-7B.
hparams = '''alg_name: "ROME"
model_name: "meta-llama/Llama-2-7b-hf"
device: 0

layers: [5]
fact_token: "subject_last"
v_num_grad_steps: 20
v_lr: 0.5
v_loss_layer: 31
clamp_norm_factor: 4
kl_factor: 0.0625
mom2_adjustment: true
context_template_length_params: [[5], [10]]
rewrite_module_tmp: "model.layers.{}.mlp.down_proj"
layer_module_tmp: "model.layers.{}"
mlp_module_tmp: "model.layers.{}.mlp"
attn_module_tmp: "model.layers.{}.self_attn"
ln_f_module: "model.norm"
lm_head_module: "lm_head"

batch_size: 1
max_length: 128
'''

open('/content/work/ROME_llama2_sanity.yaml', 'w', encoding='utf-8').write(hparams)
print('saved /content/work/ROME_llama2_sanity.yaml')



In [ ]:
# Purpose: Preflight check.
import os, torch
print('cuda=', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu=', torch.cuda.get_device_name(0))
for p in ['/content/work/ROME_llama2_sanity.yaml','/content/work/edits_bias_stress_60.json','/content/work/crows_pairs_sample.jsonl','/content/work/bbq_sample.jsonl']:
    print(p, os.path.exists(p))



In [ ]:
# Purpose: Run ROME Llama2 sanity (3 rounds, step=2).
import os, subprocess
OUT_DIR='/content/work/results/ROME_LLAMA2_SANITY_3R/rounds'
os.makedirs(OUT_DIR, exist_ok=True)

cmd=[
    'python','-u','run_edit_fairness_rounds.py',
    '--hparams','/content/work/ROME_llama2_sanity.yaml',
    '--edits_json','/content/work/edits_bias_stress_60.json',
    '--crows','/content/work/crows_pairs_sample.jsonl',
    '--bbq','/content/work/bbq_sample.jsonl',
    '--rounds','3',
    '--step','2',
    '--start_round','0',
    '--out_dir',OUT_DIR,
]

env=os.environ.copy()
env['PYTHONUNBUFFERED']='1'
env['TOKENIZERS_PARALLELISM']='false'
env['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True,max_split_size_mb:64'
env['PYTHONPATH']='/content/EasyEdit:'+env.get('PYTHONPATH','')

p=subprocess.Popen(cmd, cwd='/content/work', env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
ret=p.wait()
if ret!=0:
    raise RuntimeError(f'run failed: {ret}')



In [ ]:
# Purpose: Compute drift and draw curve.
!python /content/work/calc_drift_metrics.py --input_dir /content/work/results/ROME_LLAMA2_SANITY_3R/rounds --pattern "round_*.json" --out /content/work/results/ROME_LLAMA2_SANITY_3R/drift_metrics.json
!python /content/work/plot_drift.py --metrics_json /content/work/results/ROME_LLAMA2_SANITY_3R/drift_metrics.json --out_png /content/work/results/ROME_LLAMA2_SANITY_3R/drift_curve.png



In [ ]:
# Purpose: Zip and download outputs.
%cd /content/work/results/ROME_LLAMA2_SANITY_3R
!zip -r rome_llama2_sanity_3r.zip rounds drift_metrics.json drift_curve.png
from google.colab import files
files.download('/content/work/results/ROME_LLAMA2_SANITY_3R/rome_llama2_sanity_3r.zip')

